# Notebook 4: Expected Shortfall (CVaR)
**Author:** Niraj Neupane | github.com/nirajneupane17
**Basel III/IV:** ES at 97.5% confidence level

In [1]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy import stats
import warnings; warnings.filterwarnings('ignore')
import sys; sys.path.insert(0, '../src')
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.figsize':(13,6),'font.size':11,'axes.titlesize':13,'axes.titleweight':'bold'})

returns = pd.read_csv('../data/returns.csv', index_col='Date', parse_dates=True)
weights = np.array([0.25, 0.25, 0.20, 0.15, 0.15])
port = returns @ weights
print(f'Loaded {len(port):,} observations | {returns.index[0].date()} to {returns.index[-1].date()}')


Loaded 2,609 observations | 2015-01-01 to 2024-12-31


In [2]:
from expected_shortfall import historical_es, rolling_es
es = historical_es(port)
print('Historical Expected Shortfall:')
print(es.round(4))
es.to_csv('../results/08_expected_shortfall.csv')

Historical Expected Shortfall:
                     VaR  ES (CVaR)  ES/VaR ratio  Tail obs
confidence_level                                           
95%               0.0115     0.0163         1.417       131
97%               0.0135     0.0202         1.491        66
99%               0.0168     0.0279         1.663        27


In [3]:
res_es = rolling_es(port, window=252, confidence_level=0.975)
fig,ax=plt.subplots(figsize=(13,5))
ax.fill_between(port.index,port,0,where=port<0,color='#E24B4A',alpha=0.3,label='Daily losses')
ax.plot(res_es.index,-res_es['VaR_97pct'],color='#185FA5',linewidth=1.3,label='Rolling 97.5% VaR')
ax.plot(res_es.index,-res_es['ES_97pct'],color='#7B1FA2',linewidth=1.3,label='Rolling 97.5% ES (Basel III/IV)')
ax.set_title('Rolling 252-Day VaR vs Expected Shortfall (97.5% — Basel III/IV)')
ax.set_ylabel('Loss'); ax.legend(fontsize=9)
plt.tight_layout(); plt.savefig('../results/05_rolling_es.png',dpi=150,bbox_inches='tight')
plt.show()